# R2. The class experiment (25 min)  |  Same client, different method
Each of you is **randomly assigned a prompting condition**. We are running a between-subjects experiment on prompting, and *you* are the participants.

**Rules**
1. Your **first** run must use your assigned template **as given** (it is logged as your condition's result).
2. Then iterate **within your condition** only. Start minimal, change one thing per run, write a hypothesis each time.
3. Watch the board: the class bar chart compares conditions with 95% intervals.

**How the arena works**
- Your prompt can use placeholders: `{text}` (the item), `{labels}`, `{codebook}` (the official definitions), `{schema}` (extraction).
- The harness adds one output rule so everyone is parsed the same way. You do **not** need to ask for a specific format.
- Every run costs real money from your allowance and appears on the board. **Write your hypothesis in `note=` before you run.**
- The hidden test set stays with the instructor. Scores you see here are on the *dev* set.

In [ ]:
# ===== Run this cell first =====
import os, sys, hashlib, json
KIT_URL = "https://github.com/atsizelti/prompt_engineering.git"   # public repo with the kit
KIT_DIR = "arena_kit/prompt_arena_kit"                            # folder inside the clone
import importlib.util
if os.path.exists("arena_kit"):
    os.system("git -C arena_kit pull -q")                # always refresh a copy cloned earlier in this session
elif importlib.util.find_spec("arena") is None:
    os.system(f"git clone -q {KIT_URL} arena_kit")
if os.path.exists(KIT_DIR) and KIT_DIR not in sys.path:
    sys.path.insert(0, KIT_DIR)
import arena

NICK      = "your-nickname"      # pick a nickname (no real name needed)
PROVIDER  = "openai"             # "openai", or "mock" (practice with no key)
ENDPOINT  = "https://script.google.com/macros/s/AKfycbxdVYynFuPHaHg73kTFzL5R0TYXrPA8Tr0adMEsb0LKUS5vpDgysktWLPTTcLWrS7Jz/exec"     # the class board (already filled in: you do not need to change it)
TOKEN     = "f-BEqiAfHhAc2O-b"
ALLOWANCE = 0.09                # dollars you may spend (the whole class shares one small budget; re-running this cell does not reset it)

if os.environ.get("ARENA_MOCK"):
    PROVIDER = "mock"
if os.environ.get("ARENA_NO_BOARD"):                  # used by the instructor's offline tests
    ENDPOINT = TOKEN = ""
KEY = None
if PROVIDER != "mock":
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = os.environ.get("OPENAI_API_KEY")
try:                                                  # the board address and password, set once as Colab secrets
    from google.colab import userdata
    ENDPOINT = ENDPOINT or userdata.get("BOARD_ENDPOINT")
    TOKEN = TOKEN or userdata.get("BOARD_TOKEN")
except Exception:
    ENDPOINT = ENDPOINT or os.environ.get("BOARD_ENDPOINT", "")
    TOKEN = TOKEN or os.environ.get("BOARD_TOKEN", "")
ENDPOINT, TOKEN = "".join(ENDPOINT.split()), "".join(TOKEN.split())   # remove hidden spaces and line breaks from pasting
arena.setup(NICK, provider=PROVIDER, api_key=KEY, endpoint=ENDPOINT or None,
            token=TOKEN or None, allowance_usd=ALLOWANCE)
arena.set_round("R2")

In [ ]:
CONDS = ["zero-shot", "definitions", "few-shot-3", "few-shot-10", "edge-case-list",
         "think-step-by-step", "effort-low", "effort-high", "role-prompt", "reason-then-label"]
FORCE = None     # the instructor may set this to a condition name
MY = FORCE or CONDS[int(hashlib.md5(NICK.encode()).hexdigest(), 16) % len(CONDS)]
print("Your condition:", MY)

EX3, EX10 = arena.examples("A", 3, seed=1), arena.examples("A", 10, seed=1)

EDGE_RULES = """Rules:
1. Irony: if a snippet praises the owners while mocking strikers, it is HOSTILE.
2. A quoted opinion counts as the paper's framing only if the paper endorses it.
3. Wage figures alone are NEUTRAL_REPORT unless a loss to trade is the point.
4. Lost cargo, insurance, prices: MERCHANT_COST.
5. Pity for strikers' families counts as SYMPATHETIC even from a hostile paper.
6. OCR noise (tbe, strik-ers) should be read as the intended word.
7. Mentions of the wharf that are not about the strike (lighthouse, regatta) are OFF_TOPIC.
8. If two framings are present, choose the one the writer puts last.
9. Silence about the strikers' motives is not hostility.
10. A neutral-sounding paper can still be HOSTILE through word choice."""

BASE = "Classify this newspaper snippet by how it frames the dockworkers' strike.\nOptions: {labels}\n"
TEMPLATES = {
    "zero-shot":          BASE + "\nText: {text}",
    "definitions":        BASE + "\nCodebook:\n{codebook}\n\nText: {text}",
    "few-shot-3":         BASE + "\nCodebook:\n{codebook}\n\nExamples:\n" + EX3 + "\n\nNow classify this one.\nText: {text}",
    "few-shot-10":        BASE + "\nCodebook:\n{codebook}\n\nExamples:\n" + EX10 + "\n\nNow classify this one.\nText: {text}",
    "edge-case-list":     BASE + "\nCodebook:\n{codebook}\n\n" + EDGE_RULES + "\n\nText: {text}",
    "think-step-by-step": BASE + "\nCodebook:\n{codebook}\n\nThink step by step before answering.\n\nText: {text}",
    "effort-low":         BASE + "\nCodebook:\n{codebook}\n\nText: {text}",
    "effort-high":        BASE + "\nCodebook:\n{codebook}\n\nText: {text}",
    "role-prompt":        "You are an expert historian of Victorian labour disputes.\n" + BASE + "\nCodebook:\n{codebook}\n\nText: {text}",
    "reason-then-label":  BASE + "\nCodebook:\n{codebook}\n\nGive a one-line reason first, then the label.\n\nText: {text}",
}
PROMPT = TEMPLATES[MY]
MODEL = "gpt-5.4-nano" if MY.startswith("effort") else None   # effort arms need a reasoning-capable model; all others use the class default
print(PROMPT[:1200])

In [ ]:
# Reasoning-effort arms use the provider's reasoning_effort setting (supported by the default model).
# Only the two 'effort-*' conditions use it (on gpt-5.4-nano); every other condition runs on the class default model.
EXTRA = {"effort-low": {"reasoning_effort": "low"}, "effort-high": {"reasoning_effort": "high"}}

In [ ]:
# First run: your template exactly as given
res = arena.evaluate(PROMPT, task="A", condition=MY, model=MODEL, extra=EXTRA.get(MY), note="first run: template as given")
res.summary()

### Iterate inside your condition
Change **one** thing, write the hypothesis, run again. Re-run this cell as often as your allowance allows.

In [ ]:
PROMPT2 = PROMPT + "\n\nBe careful with irony and archaic spelling."     # <- your change here
H = "H: warning about irony will fix some HOSTILE/SYMPATHETIC confusions."
res2 = arena.evaluate(PROMPT2, task="A", condition=MY, model=MODEL, extra=EXTRA.get(MY), note=H)
res2.summary()
print(f"change in F1: {res2.metrics['macro_f1'] - res.metrics['macro_f1']:+.3f}  (95% CI width about {res.metrics['ci_high']-res.metrics['ci_low']:.2f})")

### Think
- Is your change bigger than the confidence interval? What would you need to *conclude* it helps?
- Which condition do you expect to win on the board? Why? (Write it before the reveal.)